In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# --- 1. Define Product Demand Data (High Precision) ---
# We define the data based on your preference (high precision weekly data)
products = ['A1', 'A2', 'A3', 'B1', 'B2']

demand_data = {
    'Weekly_Mean': [
        962,  # 50000 / 52
        1923, # 100000 / 52
        2500, # 130000 / 52
        1154, # 60000 / 52
        1538  # 80000 / 52
    ],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18] # From PDF P2 Table 4
}

# Create the clean Product Demand DataFrame
demand_df = pd.DataFrame(demand_data, index=products)

print("--- 1. Product Demand Data Defined (High Precision) ---")
print(demand_df)


# --- 2. Load and Clean BOM (Bill of Materials) Data ---
bom_df_dict = {
    "A1": {
        "P1": 1, "P2": 4, "P3": 0, "P4": 0, "P5": 0,
        "P6": 1, "P7": 0, "P8": 0, "P9": 0, "P10": 1,
        "P11": 2, "P12": 4, "P13": 2, "P14": 0, "P15": 0,
        "P16": 0, "P17": 2, "P18": 4, "P19": 0, "P20": 0,
    },
    "A2": {
        "P1": 2, "P2": 0, "P3": 0, "P4": 1, "P5": 1,
        "P6": 0, "P7": 4, "P8": 0, "P9": 2, "P10": 4,
        "P11": 0, "P12": 0, "P13": 0, "P14": 4, "P15": 0,
        "P16": 1, "P17": 0, "P18": 1, "P19": 2, "P20": 2,
    },
    "A3": {
        "P1": 4, "P2": 2, "P3": 0, "P4": 2, "P5": 2,
        "P6": 0, "P7": 0, "P8": 0, "P9": 2, "P10": 1,
        "P11": 0, "P12": 0, "P13": 0, "P14": 2, "P15": 0,
        "P16": 4, "P17": 0, "P18": 0, "P19": 4, "P20": 3,
    },
    "B1": {
        "P1": 4, "P2": 2, "P3": 4, "P4": 2, "P5": 0,
        "P6": 0, "P7": 0, "P8": 4, "P9": 0, "P10": 1,
        "P11": 0, "P12": 0, "P13": 0, "P14": 2, "P15": 2,
        "P16": 0, "P17": 0, "P18": 0, "P19": 4, "P20": 3,
    },
    "B2": {
        "P1": 1, "P2": 0, "P3": 1, "P4": 0, "P5": 0,
        "P6": 2, "P7": 4, "P8": 0, "P9": 0, "P10": 0,
        "P11": 4, "P12": 4, "P13": 2, "P14": 2, "P15": 0,
        "P16": 0, "P17": 2, "P18": 4, "P19": 1, "P20": 0,
    },
}

bom_df = pd.DataFrame(bom_df_dict)

print("\n--- 2. BOM Data Loaded and Converted to DataFrame ---")
print(bom_df.head())

# ---Step 1: Calculate Product Weekly Mean and SD
print("\n--- Step 1: Calculating Product Weekly SD ---")

# Calculate Weekly_SD = Weekly_Mean * Weekly_CV
demand_df['Weekly_SD'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']

print("Product Data with Calculated SD:")
print(demand_df[['Weekly_Mean', 'Weekly_SD']])


# --- Step 2: Apply Z-Score to EACH Product ("First Z") ---
print("\n--- (Method B) Step 2: Calculating 99.5% 'Peak Demand' for EACH Product ---")

# Z-Score for 99.5% Service Level
Z_SCORE = norm.ppf(0.995)
print(f"Z-Score for 99.5% Service Level: {Z_SCORE:.4f}")

# Target_Product_Capacity = E[j] + Z * sigma[j]
demand_df['Target_Product_Capacity'] = demand_df['Weekly_Mean'] + (Z_SCORE * demand_df['Weekly_SD'])

print("Calculated 'Peak Demand' for *Each Product*:")
print(demand_df[['Target_Product_Capacity']])


# --- Step 3: Aggregate 'Peak Demands' to Parts ("Then Aggregate") ---
print("\n--- Step 3: Calculating Final Part Capacity by Summing 'Peak Demands' ---")

# Get the vector of "Peak Demands" for each product
product_target_capacities = demand_df['Target_Product_Capacity']

# Create the result DataFrame for part demand
part_demand_df = pd.DataFrame(index=bom_df.index)

# Final Part Target = BOM * (Vector of Product Peak Demands)
part_demand_df['Target_Weekly_Capacity_Final'] = bom_df.dot(product_target_capacities)

# Round up, as capacity cannot be fractional
part_demand_df['Target_Weekly_Capacity_Final'] = np.ceil(
    part_demand_df['Target_Weekly_Capacity_Final']
)

# --- 4. Final Output (Method B) ---
print("\n--- FINAL TASK 1: Robust Capacity Plan (Method B: 'First Z, Then Aggregate') ---")

# (For comparison, we also add the simple weekly mean)
part_demand_df['Part_Weekly_Mean'] = bom_df.dot(demand_df['Weekly_Mean']).round(2)

# Select and format final columns
final_output = part_demand_df[['Part_Weekly_Mean', 'Target_Weekly_Capacity_Final']]

# Print the final result to the console
print(final_output)

print("\n--- Calculation Complete ---")


In [ ]:
!pip install pulp
import pandas as pd
import numpy as np
from scipy.stats import norm
# !pip install pulp  <-- Uncomment in Colab if you need to install the PuLP library
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus

# --- 1. Define Product Demand Data (High Precision) ---
# ... (This part of the code is identical to the previous file) ...
products = ['A1', 'A2', 'A3', 'B1', 'B2']
demand_data = {
    'Weekly_Mean': [962, 1923, 2500, 1154, 1538],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18]
}
demand_df = pd.DataFrame(demand_data, index=products)
print("--- 1. Product Demand Data Defined (High Precision) ---")
print(demand_df)


# --- 2. Load and Clean BOM (Bill of Materials) Data ---
# ... (This part of the code is identical to the previous file) ...
bom_df_dict = {
    "A1": {"P1": 1, "P2": 4, "P3": 0, "P4": 0, "P5": 0, "P6": 1, "P7": 0, "P8": 0, "P9": 0, "P10": 1, "P11": 2, "P12": 4, "P13": 2, "P14": 0, "P15": 0, "P16": 0, "P17": 2, "P18": 4, "P19": 0, "P20": 0},
    "A2": {"P1": 2, "P2": 0, "P3": 0, "P4": 1, "P5": 1, "P6": 0, "P7": 4, "P8": 0, "P9": 2, "P10": 0, "P11": 0, "P12": 0, "P13": 0, "P14": 4, "P15": 0, "P16": 1, "P17": 0, "P18": 1, "P19": 2, "P20": 2},
    "A3": {"P1": 4, "P2": 2, "P3": 0, "P4": 2, "P5": 2, "P6": 0, "P7": 0, "P8": 0, "P9": 2, "P10": 1, "P11": 0, "P12": 0, "P13": 0, "P14": 2, "P15": 0, "P16": 4, "P17": 0, "P18": 0, "P19": 4, "P20": 3},
    "B1": {"P1": 4, "P2": 2, "P3": 4, "P4": 2, "P5": 0, "P6": 0, "P7": 0, "P8": 4, "P9": 0, "P10": 1, "P11": 0, "P12": 0, "P13": 0, "P14": 2, "P15": 2, "P16": 0, "P17": 0, "P18": 0, "P19": 4, "P20": 3},
    "B2": {"P1": 1, "P2": 0, "P3": 1, "P4": 0, "P5": 0, "P6": 2, "P7": 4, "P8": 0, "P9": 0, "P10": 0, "P11": 4, "P12": 4, "P13": 2, "P14": 2, "P15": 0, "P16": 0, "P17": 2, "P18": 4, "P19": 1, "P20": 0},
}
bom_df = pd.DataFrame(bom_df_dict)
print("\n--- 2. BOM Data Loaded and Converted to DataFrame ---")


# --- Steps 1-3: Calculate Target Part Capacity ---
# ... (This part of the code is identical to the previous file) ...
print("\n--- (Method B) Step 1-3: Calculating Final Part Capacity ---")
demand_df['Weekly_SD'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
Z_SCORE = norm.ppf(0.995)
demand_df['Target_Product_Capacity'] = demand_df['Weekly_Mean'] + (Z_SCORE * demand_df['Weekly_SD'])
product_target_capacities = demand_df['Target_Product_Capacity']
part_demand_df = pd.DataFrame(index=bom_df.index)
part_demand_df['Target_Weekly_Capacity_Final'] = bom_df.dot(product_target_capacities)
part_demand_df['Target_Weekly_Capacity_Final'] = np.ceil(
    part_demand_df['Target_Weekly_Capacity_Final']
)
print(part_demand_df[['Target_Weekly_Capacity_Final']])
print("\n--- Calculation Complete (Part 1) ---")


# ===================================================================
# --- TASK 1 (Part 2): OPTIMIZED Machine & Operator Requirements ---
# ===================================================================

print(f"\n--- TASK 1 (Part 2): Calculating OPTIMIZED Needs for 'Parts Organization' ---")

# --- Step 5: Load Process Routing, Time Data (PDF P4, P5) ---
# ... (This part of the code is identical to the previous file) ...
process_routing_data = {
    'P1': ['B', 'A', 'B', 'C', 'D', 'I', 'J'], 'P2': ['A', 'C', 'D', 'H', 'J', None, None],
    'P3': ['B', 'D', 'C', 'I', 'J', None, None], 'P4': ['A', 'B', 'D', 'G', 'H', None, None],
    'P5': ['B', 'C', 'D', 'I', None, None, None], 'P6': ['A', 'B', 'C', 'D', 'H', 'I', 'J'],
    'P7': ['E', 'F', 'C', 'D', 'I', 'J', None], 'P8': ['E', 'H', 'J', 'I', None, None, None],
    'P9': ['F', 'G', 'E', 'G', 'I', 'J', None], 'P10': ['E', 'F', 'I', 'J', None, None, None],
    'P11': ['E', 'G', 'E', 'G', 'I', None, None], 'P12': ['E', 'G', 'F', 'I', 'J', None, None],
    'P13': ['E', 'F', 'G', 'F', 'G', 'H', 'I'], 'P14': ['E', 'F', 'G', 'H', None, None, None],
    'P15': ['E', 'G', 'F', 'H', 'J', None, None], 'P16': ['F', 'H', 'I', 'J', None, None, None],
    'P17': ['K', 'L', 'M', None, None, None, None], 'P18': ['K', 'L', 'K', 'M', None, None, None],
    'P19': ['L', 'M', 'L', 'M', None, None, None], 'P20': ['L', 'K', 'M', None, None, None]
}
process_times_data = {
    'P1': [2.5, 1.0, 2.5, 0.5, 2.5, 1.25, 2.5], 'P2': [1.25, 0.5, 2.5, 1.0, 2.5, None, None],
    'P3': [1.75, 3.0, 0.75, 1.5, 2.5, None, None], 'P4': [1.0, 2.0, 3.0, 0.25, 1.25, None, None],
    'P5': [1.5, 0.75, 3.5, 1.75, None, None, None], 'P6': [0.75, 1.25, 0.5, 3.0, 1.0, 1.25, 2.75],
    'P7': [1.0, 1.5, 0.75, 3.5, 1.25, 2.0, None], 'P8': [1.25, 2.0, 0.5, 1.0, None, None, None],
    'P9': [1.75, 0.75, 1.25, 0.5, 1.25, 3.0, None], 'P10': [1.5, 1.75, 1.25, 2.0, None, None, None],
    'P11': [1.25, 0.5, 1.25, 0.25, 0.75, None, None], 'P12': [1.0, 0.5, 1.0, 1.25, 1.25, None, None],
    'P13': [1.25, 1.25, 0.5, 1.0, 0.25, 2, 1.25], 'P14': [1.0, 1.5, 0.5, 1.75, None, None, None],
    'P15': [0.75, 0.5, 1.25, 2.5, 2.5, None, None], 'P16': [1.25, 5.0, 1.25, 2.5, None, None, None],
    'P17': [0.75, 3.0, 3.5, None, None, None, None], 'P18': [0.75, 1.25, 0.5, 3.75, None, None, None],
    'P19': [2.25, 2.5, 2.0, 3.75, None, None, None], 'P20': [2.0, 0.75, 3.0, None, None, None, None]
}
step_cols = ['Step1', 'Step2', 'Step3', 'Step4', 'Step5', 'Step6', 'Step7']
process_routing_df = pd.DataFrame.from_dict(process_routing_data, orient='index', columns=step_cols)
process_times_df = pd.DataFrame.from_dict(process_times_data, orient='index', columns=step_cols)
process_types = sorted(list(set(process_routing_df.stack().dropna().unique()))) # [A, B, C, ..., M]

# --- Step 6: Define Constants ---
# ... (This part of the code is identical to the previous file) ...
EFFICIENCY = 0.90
RELIABILITY = 0.98
SHIFTS_PER_DAY = 2
HOURS_PER_SHIFT = 8
DAYS_PER_WEEK = 5
AVAILABLE_MIN_PER_WEEK = SHIFTS_PER_DAY * HOURS_PER_SHIFT * DAYS_PER_WEEK * 60
EFFECTIVE_MIN_PER_MACHINE_WEEK = AVAILABLE_MIN_PER_WEEK * EFFICIENCY * RELIABILITY
WEEKS_PER_YEAR = 52
WORK_WEEKS_PER_YEAR = 49
COVERAGE_FACTOR = WEEKS_PER_YEAR / WORK_WEEKS_PER_YEAR

print(f"Effective Mins/Machine/Week: {EFFECTIVE_MIN_PER_MACHINE_WEEK:.2f}")
print(f"Operator Coverage Factor (52/49): {COVERAGE_FACTOR:.4f}")

# --- Step 7: Define Machine Data (PDF P6) ---
# (Machine_Type: (Cost, Staff_C1, Staff_C2, Staff_C3, Capabilities))
machine_data = {
    'A':    (150000, 1.0, 0.25, 0.0, ['A']),
    'B':    (200000, 1.0, 0.25, 0.0, ['B']),
    'C':    (250000, 1.0, 0.25, 0.0, ['C']),
    'D':    (300000, 1.0, 0.25, 0.0, ['D']),
    'AB':   (250000, 1.0, 0.25, 0.0, ['A', 'B']),
    'AC':   (280000, 1.0, 0.25, 0.0, ['A', 'C']),
    'CD':   (350000, 1.0, 0.25, 0.0, ['C', 'D']),
    'ABC':  (300000, 1.0, 0.25, 0.0, ['A', 'B', 'C']),
    'ABCD': (400000, 1.0, 0.25, 0.0, ['A', 'B', 'C', 'D']),

    'E':    (400000, 1.0, 0.0, 0.5, ['E']),
    'F':    (400000, 1.0, 0.0, 0.5, ['F']),
    'G':    (400000, 1.0, 0.0, 0.5, ['G']),
    'EF':   (450000, 1.0, 0.0, 0.5, ['E', 'F']),
    'EG':   (450000, 1.0, 0.0, 0.5, ['E', 'G']),
    'FG':   (450000, 1.0, 0.0, 0.5, ['F', 'G']),
    'EFG':  (500000, 1.0, 0.0, 0.5, ['E', 'F', 'G']),

    'H':    (1000000, 0.0, 0.0, 2.0, ['H']),
    'I':    (500000, 0.0, 0.0, 2.0, ['I']),
    'J':    (500000, 0.0, 0.0, 2.0, ['J']),
    'IJ':   (750000, 0.0, 0.0, 2.0, ['I', 'J']),
    'K':    (80000, 0.0, 0.5, 0.0, ['K']),
    'L':    (100000, 0.0, 0.5, 0.0, ['L']),
    'M':    (50000, 0.0, 0.5, 0.0, ['M']),
    'KL':   (140000, 0.0, 0.5, 0.0, ['K', 'L']),
    'KM':   (100000, 0.0, 0.5, 0.0, ['K', 'M']),
    'LM':   (120000, 0.0, 0.5, 0.0, ['L', 'M']),
    'KLM':  (150000, 0.0, 0.5, 0.0, ['K', 'L', 'M'])
}
machine_df = pd.DataFrame.from_dict(machine_data, orient='index',
                                  columns=['Cost', 'C1', 'C2', 'C3', 'Capabilities'])

# --- Step 8: Loop to calculate requirements for each Part Center ---
optimized_results = []
target_capacities = part_demand_df['Target_Weekly_Capacity_Final']

for p in target_capacities.index: # Loop through P1 to P20
    capacity = target_capacities[p]

    # 1. Calculate the "functional requirement" for this part (p) (R_A, R_B, ...)
    part_workload_min = pd.Series(0.0, index=process_types)
    for s in step_cols: # Loop through Step1 to Step7
        process_type = process_routing_df.loc[p, s]
        time_min = process_times_df.loc[p, s]

        if pd.notna(process_type) and pd.notna(time_min):
            part_workload_min[process_type] += capacity * time_min

    # This is the Right-Hand Side (RHS) of the constraint, e.g., for P1
    # R_A = 7.23, R_B = 36.14, ...
    # We use CEILING, resulting in P1: R_A=8, R_B=37, ...
    part_machine_reqs_RHS = np.ceil(part_workload_min / EFFECTIVE_MIN_PER_MACHINE_WEEK)

    # 2. Create the LP Problem
    prob = LpProblem(f"Optimize_{p}_Center", LpMinimize)

    # 3. Create the variables (X_A, X_B, ...)
    X_vars = LpVariable.dicts("X", machine_df.index, lowBound=0, cat='Integer')

    # 4. Set the Objective Function: Minimize(Cost * X)
    prob += lpSum([machine_df.loc[m, 'Cost'] * X_vars[m] for m in machine_df.index]), "Total_Cost"

    # 5. Add Constraints
    for proc_type in process_types: # Loop through process types A, B, C, ... M
        # Find all machines capable of this process
        capable_machines = [m for m in machine_df.index if proc_type in machine_df.loc[m, 'Capabilities']]

        # RHS = Right-Hand Side (e.g., 8)
        RHS = part_machine_reqs_RHS.get(proc_type, 0)

        if RHS > 0:
            # e.g., X_A + X_AB + X_AC + X_ABC + X_ABCD >= 8
            prob += lpSum([X_vars[m] for m in capable_machines]) >= RHS, f"Constraint_{proc_type}"

    # 6. Solve!
    prob.solve()

    # 7. Store results
    if LpStatus[prob.status] == 'Optimal':
        optimal_cost = prob.objective.value()

        # Get which machines were purchased
        purchased_machines = {}
        c1_frac, c2_frac, c3_frac = 0.0, 0.0, 0.0

        for m in X_vars:
            num_purchased = X_vars[m].value()
            if num_purchased > 0:
                purchased_machines[m] = int(num_purchased)
                # Accumulate fractional staff requirements
                c1_frac += num_purchased * machine_df.loc[m, 'C1']
                c2_frac += num_purchased * machine_df.loc[m, 'C2']
                c3_frac += num_purchased * machine_df.loc[m, 'C3']

        # 8. Calculate optimized staff numbers
        c1_per_shift = np.ceil(c1_frac)
        c2_per_shift = np.ceil(c2_frac)
        c3_per_shift = np.ceil(c3_frac)

        c1_hired = np.ceil(c1_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)
        c2_hired = np.ceil(c2_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)
        c3_hired = np.ceil(c3_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)

        optimized_results.append({
            'Optimal_Cost': optimal_cost,
            'Machine_Mix': purchased_machines,
            'Total_Machines': int(sum(purchased_machines.values())),
            'C1_Per_Shift': int(c1_per_shift),
            'C2_Per_Shift': int(c2_per_shift),
            'C3_Per_Shift': int(c3_per_shift),
            'Total_C1_Hired': int(c1_hired),
            'Total_C2_Hired': int(c2_hired),
            'Total_C3_Hired': int(c3_hired)
        })
    else:
        # If solver fails
        optimized_results.append({'Optimal_Cost': -1, 'Machine_Mix': 'FAILED'})

# --- Step 9: Format and print results ---
opt_results_df = pd.DataFrame(optimized_results, index=target_capacities.index)
opt_results_df.index.name = 'Part_Center'

print("\n--- Step 9: 'Parts Organization' OPTIMIZED Resource Plan (Per Part Center) ---")
print(opt_results_df)

# --- Step 10: Print totals ---
print("\n--- Step 10: 'Parts Organization' OPTIMIZED Total Requirements ---")
total_investment = opt_results_df['Optimal_Cost'].sum()
total_staff_c1 = opt_results_df['Total_C1_Hired'].sum()
total_staff_c2 = opt_results_df['Total_C2_Hired'].sum()
total_staff_c3 = opt_results_df['Total_C3_Hired'].sum()
total_machines_count = opt_results_df['Total_Machines'].sum()

print(f"Total Machine Investment (All Centers): ${total_investment:,.2f}")
print(f"Total Machines Purchased (All Types): {int(total_machines_count)}")
print(f"Total C1 Hired: {int(total_staff_c1)}")
print(f"Total C2 Hired: {int(total_staff_c2)}")
print(f"Total C3 Hired: {int(total_staff_c3)}")

print("\n--- Task 1 (Optimized) Complete ---")


In [ ]:
!pip install pulp
import pandas as pd
import numpy as np
from scipy.stats import norm
# !pip install pulp  <-- Uncomment in Colab if you need to install the PuLP library
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus

# --- 1. Define Product Demand Data (High Precision) ---
# ... (This part of the code is identical to the previous file) ...
products = ['A1', 'A2', 'A3', 'B1', 'B2']
demand_data = {
    'Weekly_Mean': [962, 1923, 2500, 1154, 1538],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18]
}
demand_df = pd.DataFrame(demand_data, index=products)
print("--- 1. Product Demand Data Defined (High Precision) ---")
print(demand_df)


# --- 2. Load and Clean BOM (Bill of Materials) Data ---
# ... (This part of the code is identical to the previous file) ...
bom_df_dict = {
    "A1": {"P1": 1, "P2": 4, "P3": 0, "P4": 0, "P5": 0, "P6": 1, "P7": 0, "P8": 0, "P9": 0, "P10": 1, "P11": 2, "P12": 4, "P13": 2, "P14": 0, "P15": 0, "P16": 0, "P17": 2, "P18": 4, "P19": 0, "P20": 0},
    "A2": {"P1": 2, "P2": 0, "P3": 0, "P4": 1, "P5": 1, "P6": 0, "P7": 4, "P8": 0, "P9": 2, "P10": 0, "P11": 0, "P12": 0, "P13": 0, "P14": 4, "P15": 0, "P16": 1, "P17": 0, "P18": 1, "P19": 2, "P20": 2},
    "A3": {"P1": 4, "P2": 2, "P3": 0, "P4": 2, "P5": 2, "P6": 0, "P7": 0, "P8": 0, "P9": 2, "P10": 1, "P11": 0, "P12": 0, "P13": 0, "P14": 2, "P15": 0, "P16": 4, "P17": 0, "P18": 0, "P19": 4, "P20": 3},
    "B1": {"P1": 4, "P2": 2, "P3": 4, "P4": 2, "P5": 0, "P6": 0, "P7": 0, "P8": 4, "P9": 0, "P10": 1, "P11": 0, "P12": 0, "P13": 0, "P14": 2, "P15": 2, "P16": 0, "P17": 0, "P18": 0, "P19": 4, "P20": 3},
    "B2": {"P1": 1, "P2": 0, "P3": 1, "P4": 0, "P5": 0, "P6": 2, "P7": 4, "P8": 0, "P9": 0, "P10": 0, "P11": 4, "P12": 4, "P13": 2, "P14": 2, "P15": 0, "P16": 0, "P17": 2, "P18": 4, "P19": 1, "P20": 0},
}
bom_df = pd.DataFrame(bom_df_dict)
print("\n--- 2. BOM Data Loaded and Converted to DataFrame ---")


# --- Steps 1-3: Calculate Target Part Capacity ---
# ... (This part of the code is identical to the previous file) ...
print("\n--- (Method B) Step 1-3: Calculating Final Part Capacity ---")
demand_df['Weekly_SD'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
Z_SCORE = norm.ppf(0.995)
demand_df['Target_Product_Capacity'] = demand_df['Weekly_Mean'] + (Z_SCORE * demand_df['Weekly_SD'])
product_target_capacities = demand_df['Target_Product_Capacity']
part_demand_df = pd.DataFrame(index=bom_df.index)
part_demand_df['Target_Weekly_Capacity_Final'] = bom_df.dot(product_target_capacities)
part_demand_df['Target_Weekly_Capacity_Final'] = np.ceil(
    part_demand_df['Target_Weekly_Capacity_Final']
)
print(part_demand_df[['Target_Weekly_Capacity_Final']])
print("\n--- Calculation Complete (Part 1) ---")


# ===================================================================
# --- TASK 1 (Part 2): OPTIMIZED Machine & Operator Requirements ---
# ===================================================================

print(f"\n--- TASK 1 (Part 2): Calculating OPTIMIZED Needs for 'Parts Organization' ---")

# --- Step 5: Load Process Routing, Time Data (PDF P4, P5) ---
# ... (This part of the code is identical to the previous file) ...
process_routing_data = {
    'P1': ['B', 'A', 'B', 'C', 'D', 'I', 'J'], 'P2': ['A', 'C', 'D', 'H', 'J', None, None],
    'P3': ['B', 'D', 'C', 'I', 'J', None, None], 'P4': ['A', 'B', 'D', 'G', 'H', None, None],
    'P5': ['B', 'C', 'D', 'I', None, None, None], 'P6': ['A', 'B', 'C', 'D', 'H', 'I', 'J'],
    'P7': ['E', 'F', 'C', 'D', 'I', 'J', None], 'P8': ['E', 'H', 'J', 'I', None, None, None],
    'P9': ['F', 'G', 'E', 'G', 'I', 'J', None], 'P10': ['E', 'F', 'I', 'J', None, None, None],
    'P11': ['E', 'G', 'E', 'G', 'I', None, None], 'P12': ['E', 'G', 'F', 'I', 'J', None, None],
    'P13': ['E', 'F', 'G', 'F', 'G', 'H', 'I'], 'P14': ['E', 'F', 'G', 'H', None, None, None],
    'P15': ['E', 'G', 'F', 'H', 'J', None, None], 'P16': ['F', 'H', 'I', 'J', None, None, None],
    'P17': ['K', 'L', 'M', None, None, None, None], 'P18': ['K', 'L', 'K', 'M', None, None, None],
    'P19': ['L', 'M', 'L', 'M', None, None, None], 'P20': ['L', 'K', 'M', None, None, None]
}
process_times_data = {
    'P1': [2.5, 1.0, 2.5, 0.5, 2.5, 1.25, 2.5], 'P2': [1.25, 0.5, 2.5, 1.0, 2.5, None, None],
    'P3': [1.75, 3.0, 0.75, 1.5, 2.5, None, None], 'P4': [1.0, 2.0, 3.0, 0.25, 1.25, None, None],
    'P5': [1.5, 0.75, 3.5, 1.75, None, None, None], 'P6': [0.75, 1.25, 0.5, 3.0, 1.0, 1.25, 2.75],
    'P7': [1.0, 1.5, 0.75, 3.5, 1.25, 2.0, None], 'P8': [1.25, 2.0, 0.5, 1.0, None, None, None],
    'P9': [1.75, 0.75, 1.25, 0.5, 1.25, 3.0, None], 'P10': [1.5, 1.75, 1.25, 2.0, None, None, None],
    'P11': [1.25, 0.5, 1.25, 0.25, 0.75, None, None], 'P12': [1.0, 0.5, 1.0, 1.25, 1.25, None, None],
    'P13': [1.25, 1.25, 0.5, 1.0, 0.25, 2, 1.25], 'P14': [1.0, 1.5, 0.5, 1.75, None, None, None],
    'P15': [0.75, 0.5, 1.25, 2.5, 2.5, None, None], 'P16': [1.25, 5.0, 1.25, 2.5, None, None, None],
    'P17': [0.75, 3.0, 3.5, None, None, None, None], 'P18': [0.75, 1.25, 0.5, 3.75, None, None, None],
    'P19': [2.25, 2.5, 2.0, 3.75, None, None, None], 'P20': [2.0, 0.75, 3.0, None, None, None, None]
}
step_cols = ['Step1', 'Step2', 'Step3', 'Step4', 'Step5', 'Step6', 'Step7']
process_routing_df = pd.DataFrame.from_dict(process_routing_data, orient='index', columns=step_cols)
process_times_df = pd.DataFrame.from_dict(process_times_data, orient='index', columns=step_cols)
process_types = sorted(list(set(process_routing_df.stack().dropna().unique()))) # [A, B, C, ..., M]

# --- Step 6: Define Constants ---
# ... (This part of the code is identical to the previous file) ...
EFFICIENCY = 0.90
RELIABILITY = 0.98
SHIFTS_PER_DAY = 2
HOURS_PER_SHIFT = 8
DAYS_PER_WEEK = 5
AVAILABLE_MIN_PER_WEEK = SHIFTS_PER_DAY * HOURS_PER_SHIFT * DAYS_PER_WEEK * 60
EFFECTIVE_MIN_PER_MACHINE_WEEK = AVAILABLE_MIN_PER_WEEK * EFFICIENCY * RELIABILITY
WEEKS_PER_YEAR = 52
WORK_WEEKS_PER_YEAR = 49
COVERAGE_FACTOR = WEEKS_PER_YEAR / WORK_WEEKS_PER_YEAR

print(f"Effective Mins/Machine/Week: {EFFECTIVE_MIN_PER_MACHINE_WEEK:.2f}")
print(f"Operator Coverage Factor (52/49): {COVERAGE_FACTOR:.4f}")

# --- Step 7: Define Machine Data (PDF P6) ---
# (Machine_Type: (Cost, Staff_C1, Staff_C2, Staff_C3, Capabilities))
machine_data = {
    'A':    (150000, 1.0, 0.25, 0.0, ['A']),
    'B':    (200000, 1.0, 0.25, 0.0, ['B']),
    'C':    (250000, 1.0, 0.25, 0.0, ['C']),
    'D':    (300000, 1.0, 0.25, 0.0, ['D']),
    'AB':   (250000, 1.0, 0.25, 0.0, ['A', 'B']),
    'AC':   (280000, 1.0, 0.25, 0.0, ['A', 'C']),
    'CD':   (350000, 1.0, 0.25, 0.0, ['C', 'D']),
    'ABC':  (300000, 1.0, 0.25, 0.0, ['A', 'B', 'C']),
    'ABCD': (400000, 1.0, 0.25, 0.0, ['A', 'B', 'C', 'D']),

    'E':    (400000, 1.0, 0.0, 0.5, ['E']),
    'F':    (400000, 1.0, 0.0, 0.5, ['F']),
    'G':    (400000, 1.0, 0.0, 0.5, ['G']),
    'EF':   (450000, 1.0, 0.0, 0.5, ['E', 'F']),
    'EG':   (450000, 1.0, 0.0, 0.5, ['E', 'G']),
    'FG':   (450000, 1.0, 0.0, 0.5, ['F', 'G']),
    'EFG':  (500000, 1.0, 0.0, 0.5, ['E', 'F', 'G']),

    'H':    (1000000, 0.0, 0.0, 2.0, ['H']),
    'I':    (500000, 0.0, 0.0, 2.0, ['I']),
    'J':    (500000, 0.0, 0.0, 2.0, ['J']),
    'IJ':   (750000, 0.0, 0.0, 2.0, ['I', 'J']),
    'K':    (80000, 0.0, 0.5, 0.0, ['K']),
    'L':    (100000, 0.0, 0.5, 0.0, ['L']),
    'M':    (50000, 0.0, 0.5, 0.0, ['M']),
    'KL':   (140000, 0.0, 0.5, 0.0, ['K', 'L']),
    'KM':   (100000, 0.0, 0.5, 0.0, ['K', 'M']),
    'LM':   (120000, 0.0, 0.5, 0.0, ['L', 'M']),
    'KLM':  (150000, 0.0, 0.5, 0.0, ['K', 'L', 'M'])
}
machine_df = pd.DataFrame.from_dict(machine_data, orient='index',
                                  columns=['Cost', 'C1', 'C2', 'C3', 'Capabilities'])

# --- Step 8: Loop to calculate requirements for each Part Center ---
optimized_results = []
target_capacities = part_demand_df['Target_Weekly_Capacity_Final']

for p in target_capacities.index: # Loop through P1 to P20
    capacity = target_capacities[p]

    # 1. Calculate the "functional requirement" for this part (p) (R_A, R_B, ...)
    part_workload_min = pd.Series(0.0, index=process_types)
    for s in step_cols: # Loop through Step1 to Step7
        process_type = process_routing_df.loc[p, s]
        time_min = process_times_df.loc[p, s]

        if pd.notna(process_type) and pd.notna(time_min):
            part_workload_min[process_type] += capacity * time_min

    # This is the Right-Hand Side (RHS) of the constraint, e.g., for P1
    # R_A = 7.23, R_B = 36.14, ...
    # We use CEILING, resulting in P1: R_A=8, R_B=37, ...
    part_machine_reqs_RHS = np.ceil(part_workload_min / EFFECTIVE_MIN_PER_MACHINE_WEEK)

    # --- START: Added for Verification ---
    print(f"\n--- Pre-Optimization Requirements for Part Center: {p} ---")
    # Filter out process types where 0 machines are needed
    required_machines = part_machine_reqs_RHS[part_machine_reqs_RHS > 0]
    if required_machines.empty:
        print("No machine requirements calculated.")
    else:
        # Using .to_string() for cleaner console output
        print(required_machines.to_string())
    # --- END: Added for Verification ---


    # 2. Create the LP Problem
    prob = LpProblem(f"Optimize_{p}_Center", LpMinimize)

    # 3. Create the variables (X_A, X_B, ...)
    X_vars = LpVariable.dicts("X", machine_df.index, lowBound=0, cat='Integer')

    # 4. Set the Objective Function: Minimize(Cost * X)
    prob += lpSum([machine_df.loc[m, 'Cost'] * X_vars[m] for m in machine_df.index]), "Total_Cost"

    # 5. Add Constraints
    for proc_type in process_types: # Loop through process types A, B, C, ... M
        # Find all machines capable of this process
        capable_machines = [m for m in machine_df.index if proc_type in machine_df.loc[m, 'Capabilities']]

        # RHS = Right-Hand Side (e.g., 8)
        RHS = part_machine_reqs_RHS.get(proc_type, 0)

        if RHS > 0:
            # e.g., X_A + X_AB + X_AC + X_ABC + X_ABCD >= 8
            prob += lpSum([X_vars[m] for m in capable_machines]) >= RHS, f"Constraint_{proc_type}"

    # 6. Solve!
    prob.solve()

    # 7. Store results
    if LpStatus[prob.status] == 'Optimal':
        optimal_cost = prob.objective.value()

        # Get which machines were purchased
        purchased_machines = {}
        c1_frac, c2_frac, c3_frac = 0.0, 0.0, 0.0

        for m in X_vars:
            num_purchased = X_vars[m].value()
            if num_purchased > 0:
                purchased_machines[m] = int(num_purchased)
                # Accumulate fractional staff requirements
                c1_frac += num_purchased * machine_df.loc[m, 'C1']
                c2_frac += num_purchased * machine_df.loc[m, 'C2']
                c3_frac += num_purchased * machine_df.loc[m, 'C3']

        # 8. Calculate optimized staff numbers
        c1_per_shift = np.ceil(c1_frac)
        c2_per_shift = np.ceil(c2_frac)
        c3_per_shift = np.ceil(c3_frac)

        c1_hired = np.ceil(c1_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)
        c2_hired = np.ceil(c2_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)
        c3_hired = np.ceil(c3_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)

        optimized_results.append({
            'Optimal_Cost': optimal_cost,
            'Machine_Mix': purchased_machines,
            'Total_Machines': int(sum(purchased_machines.values())),
            'C1_Per_Shift': int(c1_per_shift),
            'C2_Per_Shift': int(c2_per_shift),
            'C3_Per_Shift': int(c3_per_shift),
            'Total_C1_Hired': int(c1_hired),
            'Total_C2_Hired': int(c2_hired),
            'Total_C3_Hired': int(c3_hired)
        })
    else:
        # If solver fails
        optimized_results.append({'Optimal_Cost': -1, 'Machine_Mix': 'FAILED'})

# --- Step 9: Format and print results ---
opt_results_df = pd.DataFrame(optimized_results, index=target_capacities.index)
opt_results_df.index.name = 'Part_Center'

print("\n--- Step 9: 'Parts Organization' OPTIMIZED Resource Plan (Per Part Center) ---")
print(opt_results_df)

# --- Step 10: Print totals ---
print("\n--- Step 10: 'Parts Organization' OPTIMIZED Total Requirements ---")
total_investment = opt_results_df['Optimal_Cost'].sum()
total_staff_c1 = opt_results_df['Total_C1_Hired'].sum()
total_staff_c2 = opt_results_df['Total_C2_Hired'].sum()
total_staff_c3 = opt_results_df['Total_C3_Hired'].sum()
total_machines_count = opt_results_df['Total_Machines'].sum()

print(f"Total Machine Investment (All Centers): ${total_investment:,.2f}")
print(f"Total Machines Purchased (All Types): {int(total_machines_count)}")
print(f"Total C1 Hired: {int(total_staff_c1)}")
print(f"Total C2 Hired: {int(total_staff_c2)}")
print(f"Total C3 Hired: {int(total_staff_c3)}")

print("\n--- Task 1 (Optimized) Complete ---")


In [ ]:
!pip install pulp
import pandas as pd
import numpy as np
from scipy.stats import norm
# !pip install pulp  <-- Uncomment in Colab if you need to install the PuLP library
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus

# --- 1. Define Product Demand Data (High Precision) ---
# ... (This part of the code is identical to the previous file) ...
products = ['A1', 'A2', 'A3', 'B1', 'B2']
demand_data = {
    'Weekly_Mean': [962, 1923, 2500, 1154, 1538],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18]
}
demand_df = pd.DataFrame(demand_data, index=products)
print("--- 1. Product Demand Data Defined (High Precision) ---")
print(demand_df)


# --- 2. Load and Clean BOM (Bill of Materials) Data ---
# ... (This part of the code is identical to the previous file) ...
bom_df_dict = {
    "A1": {"P1": 1, "P2": 4, "P3": 0, "P4": 0, "P5": 0, "P6": 1, "P7": 0, "P8": 0, "P9": 0, "P10": 1, "P11": 2, "P12": 4, "P13": 2, "P14": 0, "P15": 0, "P16": 0, "P17": 2, "P18": 4, "P19": 0, "P20": 0},
    "A2": {"P1": 2, "P2": 0, "P3": 0, "P4": 1, "P5": 1, "P6": 0, "P7": 4, "P8": 0, "P9": 2, "P10": 0, "P11": 0, "P12": 0, "P13": 0, "P14": 4, "P15": 0, "P16": 1, "P17": 0, "P18": 1, "P19": 2, "P20": 2},
    "A3": {"P1": 4, "P2": 2, "P3": 0, "P4": 2, "P5": 2, "P6": 0, "P7": 0, "P8": 0, "P9": 2, "P10": 1, "P11": 0, "P12": 0, "P13": 0, "P14": 2, "P15": 0, "P16": 4, "P17": 0, "P18": 0, "P19": 4, "P20": 3},
    "B1": {"P1": 4, "P2": 2, "P3": 4, "P4": 2, "P5": 0, "P6": 0, "P7": 0, "P8": 4, "P9": 0, "P10": 1, "P11": 0, "P12": 0, "P13": 0, "P14": 2, "P15": 2, "P16": 0, "P17": 0, "P18": 0, "P19": 4, "P20": 3},
    "B2": {"P1": 1, "P2": 0, "P3": 1, "P4": 0, "P5": 0, "P6": 2, "P7": 4, "P8": 0, "P9": 0, "P10": 0, "P11": 4, "P12": 4, "P13": 2, "P14": 2, "P15": 0, "P16": 0, "P17": 2, "P18": 4, "P19": 1, "P20": 0},
}
bom_df = pd.DataFrame(bom_df_dict)
print("\n--- 2. BOM Data Loaded and Converted to DataFrame ---")


# --- Steps 1-3: Calculate Target Part Capacity ---
# ... (This part of the code is identical to the previous file) ...
print("\n--- (Method B) Step 1-3: Calculating Final Part Capacity ---")
demand_df['Weekly_SD'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
Z_SCORE = norm.ppf(0.995)
demand_df['Target_Product_Capacity'] = demand_df['Weekly_Mean'] + (Z_SCORE * demand_df['Weekly_SD'])
product_target_capacities = demand_df['Target_Product_Capacity']
part_demand_df = pd.DataFrame(index=bom_df.index)
part_demand_df['Target_Weekly_Capacity_Final'] = bom_df.dot(product_target_capacities)
part_demand_df['Target_Weekly_Capacity_Final'] = np.ceil(
    part_demand_df['Target_Weekly_Capacity_Final']
)
print(part_demand_df[['Target_Weekly_Capacity_Final']])
print("\n--- Calculation Complete (Part 1) ---")


# ===================================================================
# --- TASK 1 (Part 2): OPTIMIZED Machine & Operator Requirements ---
# ===================================================================

print(f"\n--- TASK 1 (Part 2): Calculating OPTIMIZED Needs for 'Parts Organization' ---")

# --- Step 5: Load Process Routing, Time Data (PDF P4, P5) ---
# ... (This part of the code is identical to the previous file) ...
process_routing_data = {
    'P1': ['B', 'A', 'B', 'C', 'D', 'I', 'J'], 'P2': ['A', 'C', 'D', 'H', 'J', None, None],
    'P3': ['B', 'D', 'C', 'I', 'J', None, None], 'P4': ['A', 'B', 'D', 'G', 'H', None, None],
    'P5': ['B', 'C', 'D', 'I', None, None, None], 'P6': ['A', 'B', 'C', 'D', 'H', 'I', 'J'],
    'P7': ['E', 'F', 'C', 'D', 'I', 'J', None], 'P8': ['E', 'H', 'J', 'I', None, None, None],
    'P9': ['F', 'G', 'E', 'G', 'I', 'J', None], 'P10': ['E', 'F', 'I', 'J', None, None, None],
    'P11': ['E', 'G', 'E', 'G', 'I', None, None], 'P12': ['E', 'G', 'F', 'I', 'J', None, None],
    'P13': ['E', 'F', 'G', 'F', 'G', 'H', 'I'], 'P14': ['E', 'F', 'G', 'H', None, None, None],
    'P15': ['E', 'G', 'F', 'H', 'J', None, None], 'P16': ['F', 'H', 'I', 'J', None, None, None],
    'P17': ['K', 'L', 'M', None, None, None, None], 'P18': ['K', 'L', 'K', 'M', None, None, None],
    'P19': ['L', 'M', 'L', 'M', None, None, None], 'P20': ['L', 'K', 'M', None, None, None]
}
process_times_data = {
    'P1': [2.5, 1.0, 2.5, 0.5, 2.5, 1.25, 2.5], 'P2': [1.25, 0.5, 2.5, 1.0, 2.5, None, None],
    'P3': [1.75, 3.0, 0.75, 1.5, 2.5, None, None], 'P4': [1.0, 2.0, 3.0, 0.25, 1.25, None, None],
    'P5': [1.5, 0.75, 3.5, 1.75, None, None, None], 'P6': [0.75, 1.25, 0.5, 3.0, 1.0, 1.25, 2.75],
    'P7': [1.0, 1.5, 0.75, 3.5, 1.25, 2.0, None], 'P8': [1.25, 2.0, 0.5, 1.0, None, None, None],
    'P9': [1.75, 0.75, 1.25, 0.5, 1.25, 3.0, None], 'P10': [1.5, 1.75, 1.25, 2.0, None, None, None],
    'P11': [1.25, 0.5, 1.25, 0.25, 0.75, None, None], 'P12': [1.0, 0.5, 1.0, 1.25, 1.25, None, None],
    'P13': [1.25, 1.25, 0.5, 1.0, 0.25, 2, 1.25], 'P14': [1.0, 1.5, 0.5, 1.75, None, None, None],
    'P15': [0.75, 0.5, 1.25, 2.5, 2.5, None, None], 'P16': [1.25, 5.0, 1.25, 2.5, None, None, None],
    'P17': [0.75, 3.0, 3.5, None, None, None, None], 'P18': [0.75, 1.25, 0.5, 3.75, None, None, None],
    'P19': [2.25, 2.5, 2.0, 3.75, None, None, None], 'P20': [2.0, 0.75, 3.0, None, None, None, None]
}
step_cols = ['Step1', 'Step2', 'Step3', 'Step4', 'Step5', 'Step6', 'Step7']
process_routing_df = pd.DataFrame.from_dict(process_routing_data, orient='index', columns=step_cols)
process_times_df = pd.DataFrame.from_dict(process_times_data, orient='index', columns=step_cols)
process_types = sorted(list(set(process_routing_df.stack().dropna().unique()))) # [A, B, C, ..., M]

# --- Step 6: Define Constants ---
# ... (This part of the code is identical to the previous file) ...
EFFICIENCY = 0.90
RELIABILITY = 0.98
SHIFTS_PER_DAY = 2
HOURS_PER_SHIFT = 8
DAYS_PER_WEEK = 5
AVAILABLE_MIN_PER_WEEK = SHIFTS_PER_DAY * HOURS_PER_SHIFT * DAYS_PER_WEEK * 60
EFFECTIVE_MIN_PER_MACHINE_WEEK = AVAILABLE_MIN_PER_WEEK * EFFICIENCY * RELIABILITY
WEEKS_PER_YEAR = 52
WORK_WEEKS_PER_YEAR = 49
COVERAGE_FACTOR = WEEKS_PER_YEAR / WORK_WEEKS_PER_YEAR

print(f"Effective Mins/Machine/Week: {EFFECTIVE_MIN_PER_MACHINE_WEEK:.2f}")
print(f"Operator Coverage Factor (52/49): {COVERAGE_FACTOR:.4f}")

# --- Step 7: Define Machine Data (PDF P6) ---
# (Machine_Type: (Cost, Staff_C1, Staff_C2, Staff_C3, Capabilities))
machine_data = {
    'A':    (150000, 1.0, 0.25, 0.0, ['A']),
    'B':    (200000, 1.0, 0.25, 0.0, ['B']),
    'C':    (250000, 1.0, 0.25, 0.0, ['C']),
    'D':    (300000, 1.0, 0.25, 0.0, ['D']),
    'AB':   (250000, 1.0, 0.25, 0.0, ['A', 'B']),
    'AC':   (280000, 1.0, 0.25, 0.0, ['A', 'C']),
    'CD':   (350000, 1.0, 0.25, 0.0, ['C', 'D']),
    'ABC':  (300000, 1.0, 0.25, 0.0, ['A', 'B', 'C']),
    'ABCD': (400000, 1.0, 0.25, 0.0, ['A', 'B', 'C', 'D']),

    'E':    (400000, 1.0, 0.0, 0.5, ['E']),
    'F':    (400000, 1.0, 0.0, 0.5, ['F']),
    'G':    (400000, 1.0, 0.0, 0.5, ['G']),
    'EF':   (450000, 1.0, 0.0, 0.5, ['E', 'F']),
    'EG':   (450000, 1.0, 0.0, 0.5, ['E', 'G']),
    'FG':   (450000, 1.0, 0.0, 0.5, ['F', 'G']),
    'EFG':  (500000, 1.0, 0.0, 0.5, ['E', 'F', 'G']),

    'H':    (1000000, 0.0, 0.0, 2.0, ['H']),
    'I':    (500000, 0.0, 0.0, 2.0, ['I']),
    'J':    (500000, 0.0, 0.0, 2.0, ['J']),
    'IJ':   (750000, 0.0, 0.0, 2.0, ['I', 'J']),
    'K':    (80000, 0.0, 0.5, 0.0, ['K']),
    'L':    (100000, 0.0, 0.5, 0.0, ['L']),
    'M':    (50000, 0.0, 0.5, 0.0, ['M']),
    'KL':   (140000, 0.0, 0.5, 0.0, ['K', 'L']),
    'KM':   (100000, 0.0, 0.5, 0.0, ['K', 'M']),
    'LM':   (120000, 0.0, 0.5, 0.0, ['L', 'M']),
    'KLM':  (150000, 0.0, 0.5, 0.0, ['K', 'L', 'M'])
}

# --- START: MODIFICATION ---
# Based on instructor feedback:
# Allow single-function (e.g., 'A') AND dual-function (e.g., 'AB') machines.
# Exclude triple/quad-function (e.g., 'ABC', 'ABCD') to prevent "unreasonable" optimization.
print("\n--- NOTE: Filtering for DEDICATED (len 1) and DUAL-FUNCTION (len 2) Machines ONLY ---")
machine_data_filtered = {
    m: d for m, d in machine_data.items() if len(m) <= 2
}
# The rest of the script will use this filtered list.

machine_df = pd.DataFrame.from_dict(machine_data_filtered, orient='index',
                                  columns=['Cost', 'C1', 'C2', 'C3', 'Capabilities'])
# --- END: MODIFICATION ---


# --- Step 8: Loop to calculate requirements for each Part Center ---
optimized_results = []
target_capacities = part_demand_df['Target_Weekly_Capacity_Final']

for p in target_capacities.index: # Loop through P1 to P20
    capacity = target_capacities[p]

    # 1. Calculate the "functional requirement" for this part (p) (R_A, R_B, ...)
    part_workload_min = pd.Series(0.0, index=process_types)
    for s in step_cols: # Loop through Step1 to Step7
        process_type = process_routing_df.loc[p, s]
        time_min = process_times_df.loc[p, s]

        if pd.notna(process_type) and pd.notna(time_min):
            part_workload_min[process_type] += capacity * time_min

    # This is the Right-Hand Side (RHS) of the constraint, e.g., for P1
    # R_A = 7.23, R_B = 36.14, ...
    # We use CEILING, resulting in P1: R_A=8, R_B=37, ...
    part_machine_reqs_RHS = np.ceil(part_workload_min / EFFECTIVE_MIN_PER_MACHINE_WEEK)

    # --- START: Added for Verification ---
    print(f"\n--- Pre-Optimization Requirements for Part Center: {p} ---")
    # Filter out process types where 0 machines are needed
    required_machines = part_machine_reqs_RHS[part_machine_reqs_RHS > 0]
    if required_machines.empty:
        print("No machine requirements calculated.")
    else:
        # Using .to_string() for cleaner console output
        print(required_machines.to_string())
    # --- END: Added for Verification ---


    # 2. Create the LP Problem
    prob = LpProblem(f"Optimize_{p}_Center", LpMinimize)

    # 3. Create the variables (X_A, X_B, ...)
    X_vars = LpVariable.dicts("X", machine_df.index, lowBound=0, cat='Integer')

    # 4. Set the Objective Function: Minimize(Cost * X)
    prob += lpSum([machine_df.loc[m, 'Cost'] * X_vars[m] for m in machine_df.index]), "Total_Cost"

    # 5. Add Constraints
    for proc_type in process_types: # Loop through process types A, B, C, ... M
        # Find all machines capable of this process
        # This list will now contain 'A', 'AB', 'AC' etc. but NOT 'ABC' or 'ABCD'
        capable_machines = [m for m in machine_df.index if proc_type in machine_df.loc[m, 'Capabilities']]

        # RHS = Right-Hand Side (e.g., 8)
        RHS = part_machine_reqs_RHS.get(proc_type, 0)

        if RHS > 0:
            # e.g., X_A + X_AB + X_AC >= 8
            prob += lpSum([X_vars[m] for m in capable_machines]) >= RHS, f"Constraint_{proc_type}"

    # 6. Solve!
    prob.solve()

    # 7. Store results
    if LpStatus[prob.status] == 'Optimal':
        optimal_cost = prob.objective.value()

        # Get which machines were purchased
        purchased_machines = {}
        c1_frac, c2_frac, c3_frac = 0.0, 0.0, 0.0

        for m in X_vars:
            num_purchased = X_vars[m].value()
            if num_purchased > 0:
                purchased_machines[m] = int(num_purchased)
                # Accumulate fractional staff requirements
                c1_frac += num_purchased * machine_df.loc[m, 'C1']
                c2_frac += num_purchased * machine_df.loc[m, 'C2']
                c3_frac += num_purchased * machine_df.loc[m, 'C3']

        # 8. Calculate optimized staff numbers
        c1_per_shift = np.ceil(c1_frac)
        c2_per_shift = np.ceil(c2_frac)
        c3_per_shift = np.ceil(c3_frac)

        c1_hired = np.ceil(c1_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)
        c2_hired = np.ceil(c2_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)
        c3_hired = np.ceil(c3_per_shift * SHIFTS_PER_DAY * COVERAGE_FACTOR)

        optimized_results.append({
            'Optimal_Cost': optimal_cost,
            'Machine_Mix': purchased_machines,
            'Total_Machines': int(sum(purchased_machines.values())),
            'C1_Per_Shift': int(c1_per_shift),
            'C2_Per_Shift': int(c2_per_shift),
            'C3_Per_Shift': int(c3_per_shift),
            'Total_C1_Hired': int(c1_hired),
            'Total_C2_Hired': int(c2_hired),
            'Total_C3_Hired': int(c3_hired)
        })
    else:
        # If solver fails
        optimized_results.append({'Optimal_Cost': -1, 'Machine_Mix': 'FAILED'})

# --- Step 9: Format and print results ---
opt_results_df = pd.DataFrame(optimized_results, index=target_capacities.index)
opt_results_df.index.name = 'Part_Center'

print("\n--- Step 9: 'Parts Organization' OPTIMIZED Resource Plan (Per Part Center) ---")
print(opt_results_df)

# --- Step 10: Print totals ---
print("\n--- Step 10: 'Parts Organization' OPTIMIZED Total Requirements ---")
total_investment = opt_results_df['Optimal_Cost'].sum()
total_staff_c1 = opt_results_df['Total_C1_Hired'].sum()
total_staff_c2 = opt_results_df['Total_C2_Hired'].sum()
total_staff_c3 = opt_results_df['Total_C3_Hired'].sum()
total_machines_count = opt_results_df['Total_Machines'].sum()

print(f"Total Machine Investment (All Centers): ${total_investment:,.2f}")
print(f"Total Machines Purchased (All Types): {int(total_machines_count)}")
print(f"Total C1 Hired: {int(total_staff_c1)}")
print(f"Total C2 Hired: {int(total_staff_c2)}")
print(f"Total C3 Hired: {int(total_staff_c3)}")

print("\n--- Task 1 (Optimized) Complete ---")

